# MNIST Recognition by NN

### Summary of 3 model performance
Overall it seems that CNN should be the best solution for image recognition, whilst Transformer (Encoder) tends to be the more comprehesive one(in terms of the overall relationship), downside is Transformer (Encoder) is not too sensitive to the orders.

Model 1: A shallow Multi-Layer Perceptron (MLP)
- Training Loss: 0.087
- Testing Accuracy: 94.42%

Model 2: Convolutional Neural Network (CNN)
- Training Loss: 0.101
- Testing Accuracy: 97.94%

Model 3: Transformer (Encoder)
- Training Loss: 0.114
- Testing Accuracy: 95.84%

In [1]:
import torch
import torch.nn as nn
from datasets import load_dataset

In [2]:
# load the dataset
mnist_dataset = load_dataset("ylecun/mnist")
train_data = mnist_dataset['train']
test_data = mnist_dataset['test']

Found cached dataset parquet (/Users/Lusa/.cache/huggingface/datasets/ylecun___parquet/mnist-7d7961f323d0d851/0.0.0/14a00e99c0d15a23649d0db8944380ac81082d4b021f398733dd84f3a6c569a7)


  0%|          | 0/2 [00:00<?, ?it/s]

In [3]:
# example check - get the first image and label
example = train_data[0]
image = example['image']
label = example['label']
print(example['image'].shape)

In [6]:
# set the format for pytorch
mnist_dataset.set_format(type='torch', columns=['image', 'label'])

In [7]:
mnist_dataset

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 60000
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 10000
    })
})

In [9]:
# set the batch
train_batch = torch.utils.data.DataLoader(train_data, batch_size=64, shuffle=True)
test_batch = torch.utils.data.DataLoader(test_data, batch_size=64, shuffle=False)

## Model selection 1: A shallow Multi-Layer Perceptron (MLP)

In [10]:
# set up MLP 
model_mlp = nn.Sequential(
    nn.Flatten(),
    #test using 128
    nn.Linear(784, 128),
    nn.ReLU(),
    # 10 categories 0-9
    nn.Linear(128, 10) 
)

optimizer = torch.optim.Adam(model_mlp.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

# training
for batch in train_batch:
    x = batch['image'].float() / 255.0
    y = batch['label']

    pred = model_mlp(x)
    loss = loss_fn(pred, y)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print(f"Traning Final Loss: {loss.item():.3f}")

# testing
model_mlp.eval()
correct = 0
total = 0

with torch.no_grad():
    for batch in test_batch:
        x = batch['image'].float() / 255.0
        y = batch['label']
        
        outputs = model_mlp(x)
        _, predicted = torch.max(outputs.data, 1) 
        total += y.size(0)
        correct += (predicted == y).sum().item()

accuracy = 100 * correct / total
print(f"Accuracy on the testing: {accuracy:.2f}%")

Traning Final Loss: 0.087
Accuracy on the testing: 94.42%


## Model selection 2: Convolutional Neural Network (CNN)

In [12]:
#set up the model
model_cnn = nn.Sequential(
    #test using 32
    nn.Conv2d(1, 32, kernel_size=3), 
    nn.ReLU(),
    nn.MaxPool2d(2),                
    nn.Conv2d(32, 64, kernel_size=3), 
    nn.ReLU(),
    nn.MaxPool2d(2),                
    nn.Flatten(),                  
    nn.Linear(64 * 5 * 5, 10) 
)

optimizer = torch.optim.Adam(model_cnn.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

# training
model_cnn.train()
for batch in train_batch:
    X = batch['image'].float().unsqueeze(1) / 255.0
    y = batch['label']

    preds = model_cnn(X)
    loss = loss_fn(preds, y)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print(f"Training Loss: {loss.item():.3f}")

# testing
model_cnn.eval()
correct = 0
total = 0

with torch.no_grad():
    for batch in test_batch:
        x = batch['image'].float().unsqueeze(1) / 255.0
        y = batch['label']
        
        outputs = model_cnn(x)
        _, predicted = torch.max(outputs.data, 1) 
        total += y.size(0)
        correct += (predicted == y).sum().item()

accuracy = 100 * correct / total
print(f"Accuracy on the testing: {accuracy:.2f}%")

Training Loss: 0.101
Accuracy on the testing: 97.94%


## Model selection 3: Transformer (Encoder)

In [15]:
# set up
# test on nhead=4/nhead=7
encoder_layer = nn.TransformerEncoderLayer(d_model=28, nhead=4, batch_first=True)
model_ecdr = nn.Sequential(
    # test on 6
    nn.TransformerEncoder(encoder_layer, num_layers=6),
    nn.Flatten(),
    nn.Linear(28 * 28, 10)
)

optimizer = torch.optim.Adam(model_ecdr.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

#traning
model_ecdr.train()
for batch in train_batch:
    x = batch['image'].float() / 255.0
    y = batch['label']

    preds = model_ecdr(x)
    loss = loss_fn(preds, y)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print(f"Training Loss: {loss.item():.3f}")
# testing
model_ecdr.eval()
correct = 0
total = 0

with torch.no_grad():
    for batch in test_batch:
        x = batch['image'].float() / 255.0
        y = batch['label']
        
        outputs = model_ecdr(x)
        _, predicted = torch.max(outputs.data, 1) 
        total += y.size(0)
        correct += (predicted == y).sum().item()

accuracy = 100 * correct / total
print(f"Accuracy on the testing: {accuracy:.2f}%")

Training Loss: 0.114
Accuracy on the testing: 95.84%
